# Session 6 — Exercises

**Before session 7.** Same house rule: **predict, then measure.** Write your prediction before you
run anything.

Answers go in the **✍️** cells, and code TODOs are marked `# TODO`. Solutions are released with
session 7.

## Setup — given, just run it

In [ ]:
import sys
import warnings
from pathlib import Path

from langchain.agents import create_agent
from langchain_core._api import LangChainBetaWarning
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

warnings.filterwarnings("ignore", category=LangChainBetaWarning)

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                        # noqa: E402  — also loads .env
from agent import GROUNDED_PROMPT, transcript      # noqa: E402
from tools import TOOLS                            # noqa: E402
from triage import build_model                     # noqa: E402

model = build_model()


def fresh_agent(**kwargs):
    """The v3 agent without summarization, on a fresh in-memory checkpointer."""
    return create_agent(model=model, tools=TOOLS, system_prompt=GROUNDED_PROMPT,
                        checkpointer=InMemorySaver(), **kwargs)


def say(agent, text, thread):
    result = agent.invoke({"messages": [{"role": "user", "content": text}]},
                          {"configurable": {"thread_id": thread}})
    print(f"customer  > {text}\nassistant > {result['messages'][-1].text}\n")
    return result


print(f"{settings.model_id} · {len(TOOLS)} tools")

## ⭐ Exercise 1 — Ticket number reality

**What this teaches:** a thread is a boundary, and that boundary is a feature.

Run the same two-turn conversation twice: once with both turns on thread `"A"`, then with turn one on
`"A"` and turn two on `"B"`.

- Turn 1: *"Hi, I'm tanvir@example.com. Where is my parcel for ORD-5003?"*
- Turn 2: *"What's the tracking number again?"*

**✍️ Prediction:** *(what does turn 2 say on thread A, and on thread B?)*

In [ ]:
T1 = "Hi, I'm tanvir@example.com. Where is my parcel for ORD-5003?"
T2 = "What's the tracking number again?"

# TODO: one fresh_agent(). Say T1 on "A", then T2 on "A". Then T2 on "B".

**✍️ Your answer (two sentences each).**

1. Why is B a stranger? Name the exact object that holds A's history, and the exact key that
   selects it.
2. Why is that a **feature** for a support desk? Answer in terms of privacy: what could go wrong if
   every ticket shared one thread?

## ⭐⭐ Exercise 2 — The debugging view

**What this teaches:** `stream.values` is the state after every step, and reading it tells you the
exact order an agent worked in.

Ask a question that needs **two lookups**: *"I'm ayesha@example.com — what's the status of my most
recent order?"* (`lookup_customer`, then `list_orders`, possibly `get_order`). Stream it with
`stream.values` and print, for each snapshot, the number of messages and the type of the last one.

**✍️ Prediction:** *(how many snapshots will print? which message types, in which order?)*

In [ ]:
Q = "I'm ayesha@example.com — what's the status of my most recent order?"
agent = fresh_agent()

# TODO: stream = agent.stream_events(..., {"configurable": {"thread_id": "DBG"}}, version="v3")
# TODO: for snapshot in stream.values: print the message count and type(last).__name__,
#       plus the tool names if the last message is an AIMessage with tool_calls.

**✍️ Your answer.**

- At which snapshot does the **first** `ToolMessage` land? What was the snapshot right before it?
- Why is there never a snapshot where an `AIMessage` with tool calls is the last message *and* the
  next one is another `AIMessage`? (Hint: section 11.1 of the notebook.)
- When would you reach for `stream.values` instead of `stream.messages`?

## ⭐⭐⭐ Exercise 3 — A `/summary` command for the human handoff

**What this teaches:** memory lives in the thread, and anything you send on the thread becomes part
of it.

When a ticket escalates, the human agent wants a short brief instead of the whole chat. Write
`handoff_summary(agent, thread_id) -> str`, which asks the agent *"Summarize this ticket so far for a
human support agent"* on the **same thread**. Then wire it into `project/chat.py` as `/summary`.

Build a two-turn ticket first, then call it.

In [ ]:
agent = fresh_agent()
say(agent, "Hi, I'm ayesha@example.com. ORD-5001 arrived with a cracked case.", "HANDOFF")
say(agent, "I'd like a refund rather than a replacement, please.", "HANDOFF")

def handoff_summary(agent, thread_id: str) -> str:
    # TODO: send the summary request on the SAME thread and return the reply text.
    ...

# TODO: print(handoff_summary(agent, "HANDOFF"))
# TODO: then print transcript(agent.get_state(...).values) — look at the last two lines.

**✍️ Your answer.**

- It works because the request is "just another turn", but look at the transcript afterwards. What
  did the handoff do to the **customer's** ticket? Why could that be a problem when the customer sends
  their next message?
- Write a second version, `handoff_summary_readonly`, that produces the summary **without writing to
  the thread**. (Hint: `agent.get_state(...)` gives you the messages, and `model` is a plain chat
  model.)
- Which version belongs in `chat.py`, and why?

## ⭐⭐ Exercise 4 — [ADV] A conversation cost meter

**What this teaches:** the bill grows with the history, and summarization bends the curve.

Run the eight-turn ticket below twice: once on a plain checkpointed agent, once with
`SummarizationMiddleware(model=model, trigger=("tokens", 1500), keep=("messages", 4))`. After every
turn record the **input tokens** for that turn (`get_usage_metadata_callback`, as in section 10) and
the running total.

**✍️ Prediction:** *(sketch both curves: per-turn input tokens, turn 1 → 8.)*

In [ ]:
TURNS = [
    "Hi, I'm ayesha@example.com. What did I order most recently?",
    "Has it shipped?",
    "What's the warranty on it?",
    "And my earlier order ORD-5001 — when was it delivered?",
    "Is the BrewMaster kettle in stock?",
    "What is the price of the AeroSound Pro headphones?",
    "Do I have any refunds in progress?",
    "Thanks — what was the first thing I asked today?",
]

from langchain.agents.middleware import SummarizationMiddleware   # noqa: E402


def meter(agent, thread: str) -> list[int]:
    """Run TURNS on one thread; return input tokens per turn, printing a running total."""
    # TODO
    ...

# TODO: plain = meter(fresh_agent(), "METER-PLAIN")
# TODO: bounded = meter(fresh_agent(middleware=[SummarizationMiddleware(...)]), "METER-SUMM")

**✍️ Your answer.**

- Where do the two curves separate, and why there?
- Is the summarized run cheaper **in total** on only eight turns? Account for the summarizer's own
  calls. (Are they in your count? Should they be?)
- How did each run answer the last question?
- One sentence for the session 15 interview: how does cost scale with conversation length, and what
  do you do about it?